In [2]:
import numpy as np
import pandas as pd
import requests
from datetime import datetime, timedelta
import math
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import johnsonsu
from scipy.optimize import minimize
import csv
import time

#1. Getting prices
def prices(startd, endd, tic):
    info = []
    key = '&apikey=ZKMMTO1ATDBLXH2K'
    ticker = '&symbol=' + str(tic)
    endpoint = 'function=TIME_SERIES_DAILY_ADJUSTED'
    size = '&outputsize=full'
    web = 'https://www.alphavantage.co/query?'
    url = web + endpoint + ticker + size + key

    r = requests.get(url)
    if r.status_code == 200:
        print('connection successful')
        data = r.json()
        r1 = data.get('Time Series (Daily)', {})
        for date, values in sorted(r1.items()):
            if startd <= date <= endd:
                info.append([tic, date, values['5. adjusted close']])
    return info


def compute_log_returns(data, freq='daily'):
    """
    Computes log returns at specified frequency.

    Parameters:
    -----------
    data : list of lists
        Price data: [[ticker, date, adjusted_close], ...]
    freq : str
        'daily', 'weekly', 'monthly', or 'yearly'

    Returns:
    --------
    list of lists: [[ticker, date, adjusted_close, log_return], ...]
    """
    sorted_data = sorted(data, key=lambda x: x[1])
    data_with_logret = []

    if freq == 'daily':
        for i in range(len(sorted_data)):
            if i == 0:
                data_with_logret.append(sorted_data[i] + [None])
            else:
                price_current = float(sorted_data[i][2])
                price_previous = float(sorted_data[i-1][2])
                log_return = np.log(price_current) - np.log(price_previous)
                data_with_logret.append(sorted_data[i] + [log_return])

    elif freq == 'weekly':
        if len(sorted_data) == 0:
            return data_with_logret
        data_with_logret.append(sorted_data[0] + [None])
        last_kept_idx = 0
        for i in range(1, len(sorted_data)):
            d30 = days_30_360(sorted_data[last_kept_idx][1], sorted_data[i][1])
            if d30 >= 7:
                price_current = float(sorted_data[i][2])
                price_previous = float(sorted_data[last_kept_idx][2])
                log_return = np.log(price_current) - np.log(price_previous)
                data_with_logret.append(sorted_data[i] + [log_return])
                last_kept_idx = i

    elif freq == 'monthly':
        if len(sorted_data) == 0:
            return data_with_logret
        data_with_logret.append(sorted_data[0] + [None])
        last_kept_idx = 0
        for i in range(1, len(sorted_data)):
            d30 = days_30_360(sorted_data[last_kept_idx][1], sorted_data[i][1])
            if d30 >= 30:
                price_current = float(sorted_data[i][2])
                price_previous = float(sorted_data[last_kept_idx][2])
                log_return = np.log(price_current) - np.log(price_previous)
                data_with_logret.append(sorted_data[i] + [log_return])
                last_kept_idx = i

    elif freq == 'yearly':
        if len(sorted_data) == 0:
            return data_with_logret
        data_with_logret.append(sorted_data[0] + [None])
        last_kept_idx = 0
        for i in range(1, len(sorted_data)):
            d30 = days_30_360(sorted_data[last_kept_idx][1], sorted_data[i][1])
            if d30 >= 360:
                price_current = float(sorted_data[i][2])
                price_previous = float(sorted_data[last_kept_idx][2])
                log_return = np.log(price_current) - np.log(price_previous)
                data_with_logret.append(sorted_data[i] + [log_return])
                last_kept_idx = i

    return data_with_logret


def download_and_export(ticker, startd, endd, freq='daily', path=None):
    """
    Downloads price data, computes log and simple returns, and exports to CSV.

    Parameters:
    -----------
    ticker : str
        Stock ticker symbol (e.g. 'NVDA')
    startd : str
        Start date 'YYYY-MM-DD'
    endd : str
        End date 'YYYY-MM-DD'
    freq : str
        'daily', 'weekly', 'monthly', or 'yearly'
    path : str
        File path for CSV output. Defaults to '{ticker}_returns.csv'

    Returns:
    --------
    list of lists: [[ticker, date, price, log_return, simple_return], ...]
    First row is a header row.
    """
    if path is None:
        path = f"{ticker}_returns.csv"

    raw = prices(startd, endd, ticker)
    with_log = compute_log_returns(raw, freq=freq)
    with_both = compute_returns(with_log)

    with open(path, mode='w', newline='') as f:
        writer = csv.writer(f)
        for row in with_both:
            writer.writerow(row)

    print(f"Exported {len(with_both)-1} rows to {path}")
    return with_both

def days_30_360(start_date, end_date):
    """
    Calculate the number of days between two dates using the 30/360 US convention.
    Dates expected as 'YYYY-MM-DD' strings or datetime.date-like objects.

    Returns an integer day-count.
    """
    if isinstance(start_date, str):
        d1 = datetime.strptime(start_date, '%Y-%m-%d').date()
    else:
        d1 = start_date
    if isinstance(end_date, str):
        d2 = datetime.strptime(end_date, '%Y-%m-%d').date()
    else:
        d2 = end_date
    d1_day = min(d1.day, 30)
    # Adjust d2 day per 30/360 US rule
    if d2.day == 31 and d1_day == 30:
        d2_day = 30
    else:
        d2_day = min(d2.day, 30)
    return 360 * (d2.year - d1.year) + 30 * (d2.month - d1.month) + (d2_day - d1_day)


In [4]:
def prices_month(startd, endd, tic):
    info = []
    key = '&apikey=ZKMMTO1ATDBLXH2K'
    ticker = '&symbol=' + str(tic)
    endpoint = 'function=TIME_SERIES_MONTHLY_ADJUSTED'
    size = '&outputsize=full'
    web = 'https://www.alphavantage.co/query?'
    url = web + endpoint + ticker + size + key

    r = requests.get(url)
    if r.status_code == 200:
        print('connection successful')
        data = r.json()
        r1 = data.get('Monthly Adjusted Time Series', {})
        for date, values in sorted(r1.items()):
            if startd <= date <= endd:
                info.append([tic, date, values['5. adjusted close']])
    return info


def monthly_log_returns(startd, endd, tic):
    raw = prices_month(startd, endd, tic)
    return compute_log_returns(raw, freq='daily')


def sigma_matrix(tickers, startd, endd):
    """
    Monthly covariance matrix of monthly log returns for the most recent 60 months.
    Returns (sigma_monthly, returns_df).
    """
    series = {}
    for i, tic in enumerate(tickers):
        if i > 0:
            time.sleep(12)  # free tier: 5 calls/min
        data = monthly_log_returns(startd, endd, tic)
        rows = [row for row in data if row[3] is not None][-60:]
        series[tic] = {row[1]: row[3] for row in rows}

    df = pd.DataFrame(series).dropna()
    return df.cov(), df


def capm_expected_returns(tickers, startd, endd, rf, market='SPY'):
    """
    Compute CAPM expected monthly returns for each ticker.

    E[R_i] = R_f_monthly + beta_i * (E[R_m] - R_f_monthly)
    beta_i  = Sigma[i, market] / Sigma[market, market]

    Fetches all tickers (including market) in one sigma_matrix call — no double fetch.
    """
    all_tickers = tickers + [market] if market not in tickers else tickers
    sigma, returns_df = sigma_matrix(all_tickers, startd, endd)

    rf_monthly = rf / 12
    E_rm_monthly = returns_df[market].mean()
    market_excess = E_rm_monthly - rf_monthly

    var_market = sigma.loc[market, market]
    betas = sigma[market].drop(market) / var_market

    return rf_monthly + betas * market_excess

In [5]:
tickers = ["JPM", "AAPL", "MSFT", "GS", "BAC"]
start_date = "2000-01-01"
end_date = "2020-12-31"
rf = 0.04  # annual risk-free rate GIVEN
rf_monthly = rf / 12
market = "SPY"


sigma, returns_df = sigma_matrix(tickers + [market], start_date, end_date)

E_rm_monthly = returns_df[market].mean()  # MONTHLY

market_excess = E_rm_monthly - rf_monthly

betas = sigma[market].drop(market) / sigma.loc[market, market]

mu = rf_monthly + betas * market_excess

print("Monthly Sigma Matrix:")
display(sigma.drop(market).drop(market, axis=1))

print("\nBetas vs SPY:")
display(betas)

print("\nCAPM Expected Monthly Returns:")
display(mu)

connection successful
connection successful
connection successful
connection successful
connection successful
connection successful
Monthly Sigma Matrix:


,JPM,AAPL,MSFT,GS,BAC
JPM,0.005209,0.002291,0.001637,0.005420,0.006199
AAPL,0.002291,0.007388,0.003010,0.003759,0.003221
MSFT,0.001637,0.003010,0.002689,0.001978,0.002295
GS,0.005420,0.003759,0.001978,0.007766,0.007002
BAC,0.006199,0.003221,0.002295,0.007002,0.008269



Betas vs SPY:


JPM     1.282005
AAPL    1.265187
MSFT    0.808656
GS      1.513021
BAC     1.645880
Name: SPY, dtype: float64


CAPM Expected Monthly Returns:


JPM     0.014091
AAPL    0.013950
MSFT    0.010119
GS      0.016030
BAC     0.017144
Name: SPY, dtype: float64

In [10]:
#CAPM so no shorts
excess = mu.values - rf
sigma_inv = np.linalg.inv(sigma.drop(market).drop(market, axis=1).values)

z = sigma_inv @ excess
T = z / z.sum()

tangent = pd.Series(T, index=mu.index)
print("Tangent Portfolio Weights:")
display(tangent)

Tangent Portfolio Weights:


JPM     0.317887
AAPL    0.180605
MSFT    0.318807
GS      0.049612
BAC     0.133089
dtype: float64

In [ ]:
#Not using CAPM AKA will have shorts


mu_hist = returns_df[tickers].mean()  # monthly

excess = mu_hist.values - rf_monthly
sigma_arr = sigma.drop(market).drop(market, axis=1).values
sigma_inv = np.linalg.inv(sigma_arr)

z = sigma_inv @ excess
T = z / z.sum()

tangent = pd.Series(T, index=mu_hist.index)
print("Tangent Portfolio Weights:")
display(tangent)

In [ ]:
start_date = "2000-01-01"  #YYYY-MM-DD
end_date = "2020-12-31"   #YYYY-MM-DD
ticker = "JPM"


#USING TIME SERIES MONTHLY
monthly_log = monthly_log_returns(start_date, end_date, ticker)

d2_log

connection successful


[['JPM', '2000-01-31', '24.5490', None],
 ['JPM', '2000-02-29', '24.2234', np.float64(-0.013352012089374643)],
 ['JPM', '2000-03-31', '26.5265', np.float64(0.09082512520128061)],
 ['JPM', '2000-04-28', '22.0630', np.float64(-0.1842422359990259)],
 ['JPM', '2000-05-31', '22.8493', np.float64(0.03501848497142701)],
 ['JPM', '2000-06-30', '21.1361', np.float64(-0.07793800326469036)],
 ['JPM', '2000-07-31', '23.0107', np.float64(0.08497684610742251)],
 ['JPM', '2000-08-31', '25.8148', np.float64(0.11498864572754508)],
 ['JPM', '2000-09-29', '21.8166', np.float64(-0.16827682295256974)],
 ['JPM', '2000-10-31', '21.6416', np.float64(-0.008053759516884806)],
 ['JPM', '2000-11-30', '17.5416', np.float64(-0.21004218554632947)],
 ['JPM', '2000-12-29', '21.6131', np.float64(0.2087244094838776)],
 ['JPM', '2001-01-31', '26.6722', np.float64(0.21032221214459224)],
 ['JPM', '2001-02-28', '22.6318', np.float64(-0.1642658275652975)],
 ['JPM', '2001-03-30', '21.7781', np.float64(-0.03845111921212041)],


In [ ]:
data = prices(start_date, end_date, ticker)

data_log = compute_log_returns(data, freq='monthly')

#MANUALLY GETTING MONTHLY

data_log

connection successful


[['JPM', '2000-01-03', '22.0423099265601', None],
 ['JPM', '2000-02-03', '25.3095465953426', np.float64(0.13821787544631814)],
 ['JPM', '2000-03-03', '24.6797742494794', np.float64(-0.025197608452923514)],
 ['JPM', '2000-04-03', '28.1420009624857', np.float64(0.13127910467550485)],
 ['JPM', '2000-05-03', '22.7146520120077', np.float64(-0.21425297721740977)],
 ['JPM', '2000-06-05', '24.2840279692009', np.float64(0.06680866967210797)],
 ['JPM', '2000-07-05', '22.0590375816588', np.float64(-0.09609646345282341)],
 ['JPM', '2000-08-07', '24.7430796413329', np.float64(0.11482345389623339)],
 ['JPM', '2000-09-07', '26.4477466293187', np.float64(0.06662512161137801)],
 ['JPM', '2000-10-09', '20.0054286745663', np.float64(-0.27916729062575474)],
 ['JPM', '2000-11-09', '20.9852475777905', np.float64(0.0478160241107517)],
 ['JPM', '2000-12-11', '20.2147103820511', np.float64(-0.03740911845493233)],
 ['JPM', '2001-01-11', '26.2501772829853', np.float64(0.2612621665457069)],
 ['JPM', '2001-02-12',